In [ ]:
"""
This file calculates the value of a client to counter 
"""

import numpy as np
from scipy import stats

def price_grid(m, g, lo=6.0, hi=6.0, per_g=40):
    """Candidate prices from m - lo*g to m + hi*g, step g/per_g."""
    n = int(round((lo + hi) * per_g)) + 1
    return np.linspace(m - lo * g, m + hi * g, n)

In [ ]:
def prior_G(q_grid, m, g, sigma_ln=0.5):
    """
    Client's prior about the dealer (i.e: the client's belief about the dealer's cost before any evidence)
    G0(q) = P(dealer's cost <= q), under c = m - g*LogNormal(0, sigma_ln).
    """
    u = (m - q_grid) / g                    # how far below m, in units of g
    print(1.0 - stats.lognorm(s=sigma_ln, scale=1.0).cdf(u))
    return 1.0 - stats.lognorm(s=sigma_ln, scale=1.0).cdf(u)

In [ ]:
def value_of_countering(r, q_grid, G_vals):
    """Best expected value from countering, and the counter achieving it.

    Returns (V, q_star).
    """
    ev = np.where(q_grid <= r, (r - q_grid) * G_vals, -np.inf)
    i = int(np.argmax(ev))
    return float(ev[i]), float(q_grid[i])

In [ ]:
def threshold(r, q_grid, G_vals):
    V, _ = value_of_countering(r, q_grid, G_vals)
    return r - V

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import yaml
from rational import (price_grid, limit_grid, prior_G, uniform_G,
                      value_of_countering, threshold, client_policy)

g = yaml.safe_load(open("config.yaml"))["calibration"]["g_points"]
m = 100.0
q = price_grid(m, g)
step = q[1] - q[0]

print(f"g = {g:.7f}   grid: {len(q)} points, step {step:.6f} = g/{g/step:.0f}")


In [ ]:
print(f"{'a':>10}{'b':>10}{'r':>10}{'q* grid':>12}{'(r+a)/2':>12}{'err (g)':>10}")
ok = True
for a_off, b_off, r_off in [(-2, 0, 1), (-3, -1, 2), (-1, 0, 0.5), (-4, -2, 3)]:
    a, b, r = m + a_off*g, m + b_off*g, m + r_off*g
    V, q_star = value_of_countering(r, q, uniform_G(q, a, b))
    expected = min(max((r + a) / 2, a), b)
    passed = abs(q_star - expected) <= step
    ok &= passed
    print(f"{a_off:>9.1f}g{b_off:>9.1f}g{r_off:>9.1f}g"
          f"{(q_star-m)/g:>11.4f}g{(expected-m)/g:>11.4f}g"
          f"{abs(q_star-expected)/g:>10.4f}   {'OK' if passed else 'FAIL'}")

print(f"\nGATE 1: {'PASS' if ok else 'FAIL'}")


In [ ]:
rs = limit_grid(m, g)
G  = prior_G(q, m, g)
theta, q_star = client_policy(rs, q, G)

mono  = bool(np.all(np.diff(theta) >= -1e-9))
below = bool(np.all(theta < rs))
slope = np.diff(theta) / np.diff(rs)

print(f"monotone in r:  {mono}")
print(f"theta < r:      {below}")
print(f"theta range:    m{(theta.min()-m)/g:+.3f}g  to  m{(theta.max()-m)/g:+.3f}g")
print(f"q* range:       m{(q_star.min()-m)/g:+.3f}g  to  m{(q_star.max()-m)/g:+.3f}g")
print(f"dtheta/dr:      {slope.min():.3f} to {slope.max():.3f}")
print(f"\nGATE 2: {'PASS' if mono and below else 'FAIL'}")
